# HW2: Equality-Constrained Optimization

**NTUT MuJoCo Course - Lecture 3**

Lecture 2 turned continuous dynamics into a discrete map. This homework is about the other
tool you need before you can control anything: **solving nonlinear equations**. Newton's
method turns a nonlinear problem into a sequence of *linear* ones, and a line search is what
keeps that sequence from diverging.

By the end you will have found a standing equilibrium for a real robot - the same
pineapple_v0 you will balance with LQR in lecture 4.

| Part | Points | Topic |
| --- | --- | --- |
| A | 10 | Line search, Newton's method, and the KKT system |
| B | 10 | Standing equilibrium for pineapple_v0 |
| C | 5 | Short answers |
| D | +5 bonus | What we dropped, and what it costs |

### Rules

- **DO NOT USE `while` LOOPS** anywhere in this homework.
- **DO NOT ALTER THE TEST CELLS.** They are marked `TEST CELL`, and we check.
- Every function you must write raises `NotImplementedError` until you replace the `# TODO`.
- Before submitting: **Kernel -> Restart Kernel and Run All Cells**.

In [ ]:
%matplotlib inline
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from dataclasses import dataclass
from IPython.display import display

import sys
from pathlib import Path
for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / 'lecture4' / 'pineapple_model.py').exists():
        sys.path.insert(0, str(_root / 'lecture4'))
        break
else:
    raise FileNotFoundError(
        'Could not locate lecture4/pineapple_model.py. Run this notebook from inside '
        'its own lecture folder in the course repository.')

np.set_printoptions(precision=6, suppress=True)
print('imports ok')

## Part A (10 pts): Line search, Newton's method, and the KKT system

We want to solve

$$
\begin{align}
\min_y \quad & f(y) \\
\text{subject to} \quad & c(y) = 0
\end{align}
$$

Form the Lagrangian with a multiplier $\lambda$ for each equality constraint,

$$\mathcal{L}(y, \lambda) = f(y) + \lambda^T c(y)$$

and a solution must make it stationary. Those are the **KKT conditions**:

$$
\begin{align}
\nabla_y \mathcal{L} = \nabla_y f(y) + \left[\frac{\partial c}{\partial y}\right]^T \lambda &= 0 \\
c(y) &= 0
\end{align}
$$

Stack the primal and dual variables into $z = [y^T, \lambda^T]^T$ and this is just a
**root-finding problem**: find $z$ with $r(z) = 0$, where $r$ is the pair of conditions
above. Everything else in this homework is machinery for solving $r(z) = 0$.

### Newton's method with a line search

Newton's method linearises the residual about the current guess and jumps to where that
linearisation vanishes:

$$\Delta z = -\left[\frac{\partial r}{\partial z}\right]^{-1} r(z_k)$$

Taking the full step is often too aggressive far from the solution, so we scale it by
$\alpha$ and require a **merit function** $\phi$ to actually decrease:

$$\phi(z_k + \alpha \Delta z) < \phi(z_k)$$

We use a **backtracking** line search: start at $\alpha = 1$, and halve it until that
inequality holds. Here $\phi(z) = \|r(z)\|$.

Write `linesearch` first. It returns the step length, and raises if it cannot find one.

In [ ]:
def linesearch(z, dz, merit_fx, max_ls_iters=10):
    """alpha = linesearch(z, dz, merit_fx)

    Return the largest alpha in 1, 1/2, 1/4, ... (at most `max_ls_iters` trials) with
    merit_fx(z + alpha * dz) < merit_fx(z).
    """
    phi0 = float(merit_fx(z))                       # given

    # DO NOT USE A WHILE LOOP
    for i in range(max_ls_iters):
        # TODO: try alpha = 0.5**i and return it as soon as the merit strictly decreases
        pass

    raise RuntimeError(f'Line search failed after {max_ls_iters} trials')

Now the solver itself. Note where the convergence test sits: **check the residual before
taking a step**, so the returned history ends on the converged point.

In [ ]:
def newtons_method(z0, res_fx, res_jac_fx, merit_fx, tol=1e-10, max_iters=50,
                   verbose=False, step_history=None):
    """Z = newtons_method(z0, res_fx, res_jac_fx, merit_fx)

    Solve res_fx(z) = 0 from the guess z0. Returns an array of every iterate,
    starting with z0 and ending with the converged point.
    """
    history = [np.array(z0, dtype=float, copy=True)]

    # DO NOT USE A WHILE LOOP
    for iteration in range(max_iters):
        z = history[-1]

        # TODO: evaluate the residual at z and take its norm.
        # hint: vector(res_fx(z)) flattens a CasADi result into a 1-D numpy array

        # TODO: if the norm is below tol, return np.asarray(history)

        # TODO: solve for the Newton step dz (do not forget the minus sign)
        # hint: np.linalg.solve(np.asarray(res_jac_fx(z)), -r)

        # TODO: find alpha with linesearch, then append z + alpha * dz to history
        # (if step_history is not None, also append alpha to it - the tests use this)
        raise NotImplementedError("newton's method not implemented")

    raise RuntimeError(f"Newton's method did not converge after {max_iters} updates")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A1)
# A warm-up root-find with no constraints: r(x) = [sin(x0), cos(x1)].
def vector(value):
    """CasADi matrix or anything array-like -> flat 1-D numpy array."""
    return np.asarray(value, dtype=float).reshape(-1)


x = ca.MX.sym('x', 2)
r = ca.vertcat(ca.sin(x[0]), ca.cos(x[1]))
residual = ca.Function('warmup', [x], [r])
jacobian = ca.Function('warmup_jac', [x], [ca.jacobian(r, x)])
merit = lambda z: np.linalg.norm(vector(residual(z)))

steps = []
Z = newtons_method([-1.742410372590328, 1.4020334125022704], residual, jacobian,
                   merit, step_history=steps, verbose=True)
assert len(Z) == 6, f'expected 6 iterates from this starting point, got {len(Z)}'
assert steps.count(0.5) == 1, f'expected exactly one halved step, got alphas {steps}'
assert merit(Z[-1]) < 1e-10, 'did not converge'

# Both failure modes must be reported, not silently ignored.
for action in [lambda: linesearch(np.zeros(1), np.ones(1), lambda _: 1.0),
               lambda: newtons_method([1.0], lambda v: v, lambda _: np.eye(1),
                                      lambda v: np.linalg.norm(v), max_iters=0)]:
    try:
        action()
    except RuntimeError:
        pass
    else:
        raise AssertionError('a solver failure was not reported')
print('\nPart A1 tests passed.')

### The KKT system

Now build the residual and its Jacobian for a *constrained* problem. Write
$A = \partial c / \partial y$ for the constraint Jacobian and $H$ for a Hessian. The
residual and the linear system Newton solves each iteration are

$$
r(z) = \begin{bmatrix} \nabla f(y) + A^T \lambda \\ c(y) \end{bmatrix},
\qquad
\frac{\partial r}{\partial z} \approx
\begin{bmatrix} H + \rho I & A^T \\ A & -\rho I \end{bmatrix}
$$

Two things deserve attention.

**The regularizer $\rho$.** The exact KKT matrix is *indefinite* - it has both positive and
negative eigenvalues - and can be singular. Adding $+\rho I$ to the primal block and
$-\rho I$ to the dual block keeps it invertible. It also changes the step, so $\rho$ should
be small.

**Which Hessian.** The true Hessian of the Lagrangian is
$\nabla^2 f + \sum_i \lambda_i \nabla^2 c_i$, which includes the curvature of the
constraints. Dropping that second term leaves $H = \nabla^2 f$, the **Gauss-Newton**
approximation. It is cheaper and always at least positive semidefinite, but it converges
more slowly. You will implement both and compare.

In [ ]:
# these are given, no TODOs here

@dataclass
class KKTProblem:
    """Everything Newton's method needs for one equality-constrained problem."""
    cost: ca.Function
    constraint: ca.Function
    residual: ca.Function
    full_newton: ca.Function
    gauss_newton: ca.Function
    n: int          # number of primal variables
    m: int          # number of equality constraints

    def merit(self, z):
        return np.linalg.norm(vector(self.residual(z)))


print('KKTProblem ready')

In [ ]:
def build_kkt(y, cost, constraint, rho):
    """problem = build_kkt(y, cost, constraint, rho)

    `y` is a CasADi symbol; `cost` and `constraint` are expressions built from it.
    Returns a KKTProblem carrying the residual and both Jacobian variants.
    """
    n, m = y.numel(), constraint.numel()
    lam = ca.MX.sym('lambda', m)                    # given: one multiplier per constraint
    z = ca.vertcat(y, lam)                          # given
    A = ca.jacobian(constraint, y)                  # given
    gradient = ca.gradient(cost, y)                 # given

    # TODO: build the KKT residual, stacking stationarity on top of primal feasibility
    residual = None

    def matrix(hessian):
        # TODO: return the regularized block matrix shown above.
        # hint: ca.blockcat([[block11, block12], [block21, block22]]),
        #       ca.MX.eye(n) and ca.MX.eye(m) for the identity blocks
        raise NotImplementedError('kkt matrix not implemented')

    # TODO: the Gauss-Newton Hessian is the Hessian of the COST alone,
    #       the full Newton Hessian is the Jacobian of the stationarity rows wrt y.
    gauss_hessian = None
    full_hessian = None

    if residual is None or gauss_hessian is None or full_hessian is None:
        raise NotImplementedError('build_kkt not implemented')
    return KKTProblem(ca.Function('cost', [y], [cost]),
                      ca.Function('constraint', [y], [constraint]),
                      ca.Function('kkt', [z], [residual]),
                      ca.Function('fn_kkt_jac', [z], [matrix(full_hessian)]),
                      ca.Function('gn_kkt_jac', [z], [matrix(gauss_hessian)]), n, m)

In [ ]:
# this cell is given, no TODOs here
# A 2-D test problem: a quadratic plus an exponential, constrained to a circle.
def toy_problem():
    x = ca.MX.sym('x', 2)
    Q = ca.DM([[1.65539, 2.89376], [2.89376, 6.51521]])
    cost = 0.5 * ca.mtimes([x.T, Q, x]) + ca.dot(ca.DM([2, -3]), x)
    cost += ca.exp(-1.3 * x[0] + 0.3 * x[1]**2)
    return build_kkt(x, cost, ca.norm_2(x) - 0.5, 1e-3)


toy = toy_problem()
print(f'toy problem: n = {toy.n} primal, m = {toy.m} constraint, z = {toy.n + toy.m}')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A2)
# Structural checks on the two KKT matrices at a random point.
rng = np.random.default_rng(0)
z = rng.standard_normal(3)
J_fn = np.asarray(toy.full_newton(z))
J_gn = np.asarray(toy.gauss_newton(z))
assert J_fn.shape == (3, 3) and J_gn.shape == (3, 3)

# The two differ ONLY in the primal block: full Newton keeps the constraint curvature.
assert np.linalg.norm(J_fn[:2, :2] - J_gn[:2, :2]) > 1e-10, \
    'full Newton and Gauss-Newton primal blocks should differ'
np.testing.assert_allclose(J_fn[:2, 2], J_gn[:2, 2], atol=1e-12)   # A^T block
np.testing.assert_allclose(J_fn[2, :2], J_gn[2, :2], atol=1e-12)   # A block
# The dual block is exactly -rho.
np.testing.assert_allclose([J_fn[2, 2], J_gn[2, 2]], [-1e-3, -1e-3], atol=1e-12)
# Off-diagonal blocks are transposes of each other.
np.testing.assert_allclose(J_gn[:2, 2], J_gn[2, :2], atol=1e-12)
print('Part A2 tests passed.')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A3)
# do not change the starting point or tolerance in this cell
histories = {}
for name, jac in [('Full Newton', toy.full_newton), ('Gauss-Newton', toy.gauss_newton)]:
    Z = newtons_method([-0.1, 0.5, 0.0], toy.residual, jac, toy.merit,
                       tol=1e-4, max_iters=100)
    histories[name] = Z
    print(f'{name:14} {len(Z) - 1} updates, |r| = {np.linalg.norm(vector(toy.residual(Z[-1]))):.3e}, '
          f'cost = {float(toy.cost(Z[-1][:2])):.8g}')

for name, Z in histories.items():
    assert np.linalg.norm(vector(toy.residual(Z[-1]))) < 1e-4, name
    assert abs(float(toy.constraint(Z[-1][:2]))) < 1e-4, f'{name} is off the constraint'
# Both find the same minimum, but curvature information pays for itself.
np.testing.assert_allclose(histories['Full Newton'][-1][:2],
                           histories['Gauss-Newton'][-1][:2], atol=1e-3)
assert len(histories['Full Newton']) < 6, 'full Newton should converge in under 6 iterates'
assert len(histories['Gauss-Newton']) < 10, 'Gauss-Newton should converge in under 10'
print('\nPart A3 tests passed.')

fig, ax = plt.subplots(figsize=(8, 5))
for name, Z in histories.items():
    ax.semilogy(range(len(Z)), [np.linalg.norm(vector(toy.residual(z))) for z in Z],
                'o-', label=f'{name} ({len(Z) - 1} updates)')
ax.set(xlabel='Iteration', ylabel='||KKT residual||', title='Toy problem convergence')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Part B (10 pts): Standing equilibrium for pineapple_v0

Now the same solver, on a real robot. pineapple_v0 is a two-wheeled biped. **You give it a
target base height, and it gives you back the joint angles, motor torques and ground forces
that hold the robot still at that height** - which is exactly the configuration lecture 4
will linearize and balance.

At rest there are no velocities and no accelerations, so the equations of motion collapse
from $M(q)\ddot q + b(q, \dot q) = B\tau + J^T f$ down to a static force balance:

$$\nabla V(q) - B \tau - J(q)^T f = 0$$

Only masses, centers of mass and kinematics enter - no inertias, no Coriolis terms. The
given `PineappleModel` differentiates the potential energy $V$ and the contact kinematics
symbolically, so `model.evaluate(q)` hands you $\nabla V$, $J$, and the contact points.

Our variables are $y = [q, \tau, f] \in \mathbb{R}^{24}$: a 12-DOF configuration (free base
plus six joints), six motor torques, and two 3-D contact forces.

In [ ]:
# this cell is given, no TODOs here
from pineapple_model import (PineappleModel, DEFAULT_JOINT_POSITIONS, JOINT_NAMES,
                             WHEEL_BODIES)

model = PineappleModel()
print(f'model       : {model.path.name}')
print(f'total mass  : {model.mass:.4f} kg over {len(model.body_names)} bodies')
print(f'joints      : {", ".join(JOINT_NAMES)}')
print(f'wheel radius: {model.radii[0]} m, friction {model.friction[0]}')
print(f'torque limit: {model.torque_limits}')

# evaluate(q) -> gradient of the potential, contact Jacobian, contact points, com, axles
gravity, J, contacts, com, wheels = model.evaluate(model.q_guess)
print(f'\nat the reference stance: grad V is {gravity.shape}, contact Jacobian is {J.shape}')
print(f'center of mass {np.round(vector(com), 4)}')

### The optimization problem

Many configurations balance, so we pick one by **tracking a reference stance** and let the
constraints pin down the rest. For a commanded height $h$:

$$
\begin{align}
\min_{q, \tau, f} \quad & \tfrac{1}{2}(q_{1:6} - q^{ref}_{1:6})^T W (q_{1:6} - q^{ref}_{1:6})
 + \tfrac{1}{2} \cdot 10 \, \|q_{7:12} - q^{ref}_{joints}\|^2
 + \tfrac{1}{2} 10^{-3} \left(\|\tau\|^2 + \|f\|^2\right) \\
\text{subject to} \quad
 & \nabla V(q) - B\tau - J(q)^T f = 0 && \text{(12) static balance} \\
 & [\text{contact}_L]_z = [\text{contact}_R]_z = 0 && \text{(2) wheels on the ground} \\
 & \tfrac{1}{2}\left([\text{contact}_L]_{xy} + [\text{contact}_R]_{xy}\right) = p^{ref}_{xy} && \text{(2) do not walk away} \\
 & \text{yaw} = 0,\quad q_z = h && \text{(2)} \\
 & \theta_{L,wheel} = \theta_{R,wheel} = 0 && \text{(2) encoders zeroed}
\end{align}
$$

with $W = \mathrm{diag}(100, 100, 100, 1000, 1000, 1000)$ and $B = [0_{6\times6}; I_6]$,
since the six motors drive the joints and not the floating base.

That is 20 equality constraints on 24 variables. Both functions below take the target
height (through `reference`, whose third entry is $h$), so the same code solves for any
commanded height.

**Watch out:** write the quadratic terms with `ca.sumsqr`, not `ca.norm_2(...)**2` - the
2-norm is not differentiable at zero and our initial guess sits exactly there, which would
hand you `NaN` on the very first iteration.

In [ ]:
def pineapple_cost(y, reference):
    """Scalar cost. y = [q(12), tau(6), forces(6)]; reference is the 12-vector to track."""
    q, tau, forces = y[:12], y[12:18], y[18:]
    weights = ca.DM([100, 100, 100, 1000, 1000, 1000])
    # TODO: weighted base pose tracking + joint tracking (weight 10)
    #       + 1e-3 regularization on torques and forces.
    # hint: ca.dot(weights, (q[:6] - reference[:6])**2) and ca.sumsqr(...)
    raise NotImplementedError('pineapple cost not implemented')


def pineapple_constraint(y, target_height):
    """The 20 equality constraints, stacked into one column."""
    q, tau, forces = y[:12], y[12:18], y[18:]
    gravity, J, contacts, _, _ = model.evaluate(q)        # given

    # TODO: static balance - gravity, minus the motor torques acting on the last six
    #       coordinates only, minus the contact wrench J^T f
    # hint: ca.vertcat(ca.MX.zeros(6), tau) puts tau in the actuated block

    # TODO: stack balance (12) with
    #       contacts[2, :].T                                           -> wheels on the ground
    #       (contacts[:2, 0] + contacts[:2, 1]) / 2 - model.contact_midpoint_xy
    #       q[5]                                                       -> zero yaw
    #       q[2] - target_height
    #       q[8], q[11]                                                -> wheel encoders
    raise NotImplementedError('pineapple constraint not implemented')

In [ ]:
# this cell is given, no TODOs here
# rho is fixed at 1e-6 on purpose: at 1e-3 Gauss-Newton does not converge on this problem.
PINEAPPLE_RHO = 1e-6


def solve_standing(target_height, full_newton=True, verbose=False):
    """Solve for the pose that holds the robot still at `target_height` metres.

    Returns a dict with the full configuration, a joint-name -> angle mapping, the
    motor torques, the ground forces, and the iterate history.
    """
    reference = model.q_guess.copy()
    reference[2] = target_height

    y = ca.MX.sym('y', 24)
    problem = build_kkt(y, pineapple_cost(y, reference),
                        pineapple_constraint(y, target_height), PINEAPPLE_RHO)

    # Start from the reference stance, zero torques, each wheel holding half the weight.
    normal = -model.mass * model.gravity[2] / 2
    z0 = np.r_[reference, np.zeros(6), [0, 0, normal] * 2, np.zeros(problem.m)]
    jacobian = problem.full_newton if full_newton else problem.gauss_newton
    Z = newtons_method(z0, problem.residual, jacobian, problem.merit,
                       tol=1e-8, max_iters=60, verbose=verbose)

    z = Z[-1]
    return {'target_height': target_height,
            'q': z[:12],
            'joints': dict(zip(JOINT_NAMES, z[6:12])),
            'torques': dict(zip(JOINT_NAMES, z[12:18])),
            'forces': z[18:24].reshape(2, 3),
            'iterates': len(Z),
            'residual': float(np.linalg.norm(vector(problem.residual(z)))),
            'problem': problem,
            'history': Z}


print('reachable heights are roughly 0.20 - 0.38 m; past that the legs run out of travel')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B)
# do not change rho or these heights
for height in (0.22, 0.26, 0.30):
    result = solve_standing(height)
    q, forces = result['q'], result['forces']
    print(f'h = {height:.2f} m: {result["iterates"]} iterates, |r| = {result["residual"]:.2e}, '
          f'thigh {result["joints"]["L_thigh_joint"]:+.4f} rad')
    assert result['residual'] < 1e-8, f'did not converge at {height} m'
    np.testing.assert_allclose(q[2], height, atol=1e-8)          # commanded height met
    np.testing.assert_allclose(q[[8, 11]], 0, atol=1e-8)         # wheel encoders zeroed
    # The two wheels together must carry exactly the robot's weight.
    np.testing.assert_allclose(forces.sum(axis=0) + model.mass * model.gravity, 0, atol=1e-6)
    # Left and right legs take almost the same angles. Only almost: the battery and
    # the base center of mass are slightly offset in y, so the robot is not perfectly
    # symmetric and the two legs differ by a few times 1e-5 rad.
    np.testing.assert_allclose(q[6:8], q[9:11], atol=1e-3)

# A taller command must straighten the legs.
low, high = solve_standing(0.22), solve_standing(0.34)
assert high['joints']['L_thigh_joint'] < low['joints']['L_thigh_joint'], \
    'standing taller should reduce the thigh angle'

# Same equilibrium from both Hessians, but full Newton gets there sooner.
full = solve_standing(0.30, full_newton=True)
gauss = solve_standing(0.30, full_newton=False)
np.testing.assert_allclose(full['q'], gauss['q'], atol=1e-6)
assert full['iterates'] < 10, f'full Newton took {full["iterates"]} iterates'
assert gauss['iterates'] < 30, f'Gauss-Newton took {gauss["iterates"]} iterates'
assert full['iterates'] < gauss['iterates'], 'full Newton should need fewer iterations here'
print(f'\nat 0.30 m: full Newton {full["iterates"]} iterates, Gauss-Newton {gauss["iterates"]}')
print('\nPart B tests passed.')

### Your robot, your height

Change `TARGET_HEIGHT` below and re-run to get the joint states for any commanded height.
Anywhere from about **0.20 m to 0.38 m** works; past that the legs run out of travel and
Newton's method fails to converge rather than returning a pose the robot cannot hold.

In [ ]:
# this cell is given - change TARGET_HEIGHT and re-run
TARGET_HEIGHT = 0.30

standing = solve_standing(TARGET_HEIGHT)
print(f'Standing at {TARGET_HEIGHT:.3f} m  ({standing["iterates"]} Newton iterates, '
      f'|r| = {standing["residual"]:.2e})\n')
print(f'{"joint":<16}{"angle (rad)":>13}{"angle (deg)":>13}{"torque (Nm)":>13}')
for name in JOINT_NAMES:
    angle = standing['joints'][name]
    print(f'{name:<16}{angle:>13.6f}{np.rad2deg(angle):>13.3f}'
          f'{standing["torques"][name]:>13.4f}')
print(f'\nbase position (m) : {np.round(standing["q"][:3], 6)}')
print(f'base rpy (rad)    : {np.round(standing["q"][3:6], 6)}')
print(f'ground forces (N) : left {np.round(standing["forces"][0], 3)}, '
      f'right {np.round(standing["forces"][1], 3)}')
print('\ntarget_dof_pos = [' + ', '.join(f'{standing["joints"][n]:.10f}' for n in JOINT_NAMES) + ']')

Standing at 0.300 m  (6 Newton iterates, |r| = 8.55e-09)

joint             angle (rad)  angle (deg)  torque (Nm)
L_thigh_joint        0.867862       49.725      -0.1561
L_calf_joint        -1.333593      -76.409       2.0634
L_wheel_joint        0.000000        0.000       0.0000
R_thigh_joint        0.867855       49.724      -0.1547
R_calf_joint        -1.333599      -76.410       2.0632
R_wheel_joint        0.000000        0.000       0.0002

base position (m) : [ 0.017364 -0.0762    0.3     ]
base rpy (rad)    : [ 0.       -0.014172 -0.      ]
ground forces (N) : left [-0.    -0.    31.577], right [-0.     0.    31.567]

target_dof_pos = [0.8678622410, -1.3335933218, 0.0000000000, 0.8678552104, -1.3335992666, 0.0000000000]


### See it

The cell below renders solved poses in MeshCat using the robot's real visual meshes. Green
spheres mark where the wheels touch the ground; the orange sphere is the center of mass,
with a line dropped to the floor. At a valid equilibrium it lands between the contacts.

Three heights are drawn side by side so you can watch the legs fold as the command drops.
MeshCat is optional - if it is not installed this cell just says so and nothing else in the
notebook is affected.

In [ ]:
# this cell is given, no TODOs here
# Only the import and the server launch are guarded - if your own code is not
# finished yet, that error should reach you rather than be reported as a MeshCat problem.
try:
    from pineapple_viz import create_meshcat_viewer, show_poses
    viewer = create_meshcat_viewer()
except (ImportError, RuntimeError) as exc:
    viewer = None
    print(exc)

if viewer is not None:
    show_poses(viewer, model,
               [(solve_standing(h)['q'], f'{h:.2f} m') for h in (0.22, 0.30, 0.36)])
    print(f'MeshCat is also open at {viewer.url()}')
    display(viewer.jupyter_cell())

Two things to take away.

**Full Newton wins where the constraints curve.** On the toy problem it was roughly 4
iterations against 9; here it is about 6 against 20. The constraint curvature term
$\sum_i \lambda_i \nabla^2 c_i$ that Gauss-Newton throws away is exactly the kinematics of
the robot, and the multipliers $\lambda$ multiplying it are large because they carry the
robot's weight.

**Regularization is not free.** This problem is given to you with $\rho = 10^{-6}$. At
$\rho = 10^{-3}$ - the value that works fine on the toy problem - Gauss-Newton does not
converge here at all. A regularizer big enough to guarantee an invertible matrix can be big
enough to destroy the step it was protecting.

## Part C (5 pts): Short answer

Answer in one or two sentences each.

**1.** Why do we need a line search at all? What goes wrong if you always take $\alpha = 1$?

**PUT YOUR ANSWER HERE**

**2.** Do we need a line search for both convex and nonconvex problems?

**PUT YOUR ANSWER HERE**

**3.** Name one case where a line search is definitely unnecessary.

**PUT YOUR ANSWER HERE**

**4.** The KKT matrix is indefinite, so we add $+\rho I$ to the primal block and $-\rho I$
to the dual block. Why the opposite signs?

**PUT YOUR ANSWER HERE**

## Part D (5 pts BONUS): What we dropped

Newton's method solves *equalities*. But the real problem has inequalities too: wheels can
only push on the ground and never pull, the contact force has to stay inside the friction
cone, the motors saturate, and the joints have travel limits.

We ignored every one of them. That is only legitimate if they turn out to be **inactive** -
satisfied with room to spare at the solution. Check that.

In [ ]:
def check_dropped_inequalities(result):
    """Return a dict of name -> bool for each constraint we left out of the solve.

    `result` is whatever solve_standing returned.
    """
    q = result['q']
    tau = np.array([result['torques'][name] for name in JOINT_NAMES])
    forces = result['forces']
    # TODO: fill in each check.
    #   'normal forces positive' : every forces[:, 2] > 0
    #   'inside friction cone'   : ||forces[:, :2]|| <= model.friction * forces[:, 2]
    #   'torque limits'          : |tau| <= model.torque_limits
    #   'joint limits'           : model.joint_limits[name] brackets q[6 + i]
    #                              (JOINT_NAMES gives the order; a joint missing from
    #                               the dict is unlimited)
    raise NotImplementedError('dropped inequality check not implemented')

In [45]:
# ------------------------------------------------------------------- TEST CELL (D)
verdict = check_dropped_inequalities(standing)
for name, ok in verdict.items():
    print(f'  {name:24} {"satisfied" if ok else "VIOLATED"}')
assert set(verdict) == {'normal forces positive', 'inside friction cone',
                        'torque limits', 'joint limits'}, 'unexpected keys'
assert all(verdict.values()), 'an inequality we dropped is actually violated'

# It must also hold across the range of heights, not just the one you happened to pick.
for height in (0.22, 0.26, 0.30, 0.34):
    assert all(check_dropped_inequalities(solve_standing(height)).values()), \
        f'an inequality is violated at {height} m'

# How much room is left? If any of these were tight, ignoring them would be wrong.
tau = np.array([standing['torques'][name] for name in JOINT_NAMES])
print(f'\nsmallest normal force  : {standing["forces"][:, 2].min():.3f} N')
print(f'largest torque demand  : {np.abs(tau).max():.3f} Nm of '
      f'{model.torque_limits.min():.2f} available')
print('\nPart D tests passed.')

  normal forces positive   satisfied
  inside friction cone     satisfied
  torque limits            satisfied
  joint limits             satisfied

smallest normal force  : 31.567 N
largest torque demand  : 2.737 Nm of 3.69 available

Part D tests passed.


Every dropped inequality is slack, so the equality-only solve was justified *for this
problem at this height*. Push the target height far enough and that stops being true - the
legs run out of travel or the motors saturate - and then Newton's method will happily
return a configuration the robot cannot actually hold.

Handling inequalities properly needs an interior-point or active-set method, which is where
a follow-up lecture would go. Lecture 4 takes this equilibrium and asks a different
question: once the robot is standing here, how do we keep it from falling over?

### Submitting

1. **Kernel -> Restart Kernel and Run All Cells.**
2. Check that every test cell printed its `... tests passed.` line.
3. Check that you answered all four short-answer questions in Part C.
4. Export to PDF: `File -> Save and Export Notebook As -> HTML`, then print to PDF.
5. Submit the PDF **and** the `.ipynb`.